# 01 SDXL Base 1.0 でもう一度同じことをやる — SD1.5 との対比

このノートブックでは、[00ノートブック (00_sd15_intro.ipynb)](00_sd15_intro.ipynb) で SD1.5 に対して行ったのと ほぼ同じ手順を [stabilityai/stable-diffusion-xl-base-1.0](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0) (以下 SDXL Base) で繰り返します。同じ prompt / 同じ seed / [00ノートブック](00_sd15_intro.ipynb) と対応する 1〜7 章で SDXL Base の基本構造を確認し、8〜10 章では SDXL 系の派生例として Lightning, Animagine XL, style LoRA stacking を試します。**「モデルが SD1.5 → SDXL Base に変わると、どこが変わるか + SDXL エコシステムの派生方式」**を直接見比べられるように構成しています。

## 0. はじめに

### このノートブックでやること

#### 1〜7 章: SDXL Base の基本構造 ([00ノートブック](00_sd15_intro.ipynb) と対応)

1. [環境確認](#env) — device (CPU / MPS / CUDA) とパッケージを確認
2. [モデル読み込み](#load) — `StableDiffusionXLPipeline` を Hugging Face から読み込む
3. [画像生成](#gen) — [00ノートブック](00_sd15_intro.ipynb) と同じ prompt で 1 枚生成
4. [パイプラインを覗いてみる](#peek) — components 一覧、パラメータ数、disk バイト数
5. [パイプラインの中身を見る](#inside) — text_encoder × 2 / UNet / VAE / scheduler
6. [tokenizer の動作を確認](#tokenizer) — 2 つの tokenizer が同じ prompt をどう扱うか
7. [seed と steps を変えてみる](#sweep) — 同じ仕組み、SDXL ネイティブのスケール感で

#### 8〜10 章: SDXL 派生方式の試行 (01 固有)

8. [efficiency LoRA: SDXL Lightning](#lightning) — 蒸留 distilled 版、4 steps で動く LoRA ([HF model card](https://huggingface.co/ByteDance/SDXL-Lightning))
9. [anime full FT model: Animagine XL 3.1](#animagine) — Anything-v4 相当の SDXL 版 ([HF model card](https://huggingface.co/cagliostrolab/animagine-xl-3.1))
10. [style LoRA stacking](#loras) — Animagine XL に [Linaqruf](https://huggingface.co/Linaqruf) 製の style LoRA を 3 種類 (style-enhancer / anime-detailer / anime-nouveau) 重ねて画風差を比較
11. [まとめ](#summary)

### 生成画像の保存

このノートブックで生成した画像は、notebook 内表示に加えて `outputs/01_sdxl_base_intro/` ディレクトリに PNG として保存されます (§3, §7, §8, §9, §10 で計 9 枚)。後から外部ビューア / image diff ツールで比較したい場合に便利です。

### SDXL Base とは — SD1.5 との違いを 1 表で

[SDXL Base 1.0](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0) は 2023 年に Stability AI から release された、SD1.5 と同じ **latent diffusion model** の系譜にある後継モデルです。アーキテクチャの骨格 (CLIP text encoder + UNet + VAE + scheduler) は SD1.5 と同じですが、各部品が大幅に拡張されています:

| 項目 | SD1.5 (00) | SDXL Base 1.0 (01) |
|---|---|---|
| text encoder | CLIP ViT-L/14 × 1 (~123 M) | **CLIP ViT-L/14 + OpenCLIP ViT-bigG/14 × 2** (~817 M) |
| UNet | 約 860 M, 64×64 latent | **約 2.6 B, 128×128 latent** |
| VAE | AutoencoderKL (84 M) | AutoencoderKL (84 M、SDXL 用に再学習) |
| ネイティブ解像度 | 512×512 | **1024×1024** |
| cross_attention_dim | 768 | **2048** (CLIP-L 768 + OpenCLIP-bigG 1280 を concat) |
| 全パラメータ | 約 1.37 B | **約 3.5 B** |
| safety_checker | あり (~304 M) | **なし** |
| disk cache (fp32 既定 variant) | 約 5 GB | **約 14 GB** |
| MPS の罠 | fp16 で safety_checker 誤発火 | **fp16 で VAE 等が NaN** → fp32 必須 (詳細は §4) |

ざっくり言えば「2 つの text encoder を concat した強い条件付け + ほぼ 3 倍重い UNet + 4 倍解像度」のモデルです。

### 情報源 (現用 → 歴史的)

- HF model card (本ノートブックで使う 重み): <https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0> — SDXL Base 1.0 の重み・config・2 つの tokenizer がここに集約されている
- PyPI (本ノートブックで使う library の配布元): <https://pypi.org/project/diffusers/> — [00ノートブック](00_sd15_intro.ipynb) と同じ。`pip install diffusers` の wheel 配布元
- GitHub (本ノートブックで使う library のソースコード): <https://github.com/huggingface/diffusers> — `StableDiffusionXLPipeline` の実装本体
- GitHub (公式 reference 実装): <https://github.com/Stability-AI/generative-models> — Stability AI 公式の SDXL training/inference code (Stable Diffusion 3 系もここ)
- 論文: <https://arxiv.org/abs/2307.01952> — "SDXL: Improving Latent Diffusion Models for High-Resolution Image Synthesis" (Podell et al., 2023)

### [00ノートブック](00_sd15_intro.ipynb) から繰り返す前提

Hugging Face / `from_pretrained` / latent diffusion の基礎・cache の仕組みは [00ノートブック](00_sd15_intro.ipynb) で説明済なので、このノートブックでは SDXL 固有の差分にフォーカスして説明します。[00ノートブック](00_sd15_intro.ipynb) を未読の場合は先にそちらを通すのが推奨です。


## 0. 環境セットアップ（3環境 自動切替: Colabだけ pip / path も自動）


In [ ]:
# 3環境(Mac/Win/Colab)を同一ファイルで動かすための判定。Colabのみ pip（Mac/Winはenvに在るのでskip）。
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "transformers==5.9.0" "accelerate==1.13.0"
    !pip uninstall -q -y torchao   # Colab同梱の古いtorchao(0.10)がdiffusers0.38と非互換→除去（§8 SDXL Lightning LoRAで発火）
    print("Colab: pip done")
else:
    print("local(Mac/Win): pip skip（env利用）")

<a id="env"></a>

---
## 1. 環境確認

00 と同じ手順。device と dtype を確認します。

**SDXL Base 特有の注意**: MPS + fp16 / bf16 だと **UNet または VAE で NaN が出る既知問題**があるため、MPS では fp32 を選びます (CUDA は fp16、CPU は fp32)。

In [ ]:
import sys
import logging
import torch
import diffusers
import transformers

logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Diffusers   :", diffusers.__version__)
print("Transformers:", transformers.__version__)

# device と dtype を選ぶ
# - cuda → fp16 (NVIDIA GPU)
# - mps  → fp32 (SDXL は fp16 だと UNet/VAE で NaN が出る既知問題があるため)
# - cpu  → fp32
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float32
else:
    device = "cpu"
    dtype = torch.float32

print("device      :", device)
print("dtype       :", dtype)

<a id="load"></a>

---
## 2. モデル読み込み

`StableDiffusionXLPipeline` を Hugging Face cache から読み込みます。今回 load する HF Hub レポジトリは [`stabilityai/stable-diffusion-xl-base-1.0`](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0) です。

**初回実行は SD1.5 よりさらに時間がかかります**:
- text_encoder + text_encoder_2 + UNet + VAE で **合計 約 14 GB** を download (fp32 のデフォルト variant の場合。`variant="fp16"` を指定すれば約 7 GB に半減)
- 回線速度次第で **十数分〜1 時間** 程度
- 保存先は `~/.cache/huggingface/hub/` (00 と同じ)
- **2 回目以降は cache から読むので数秒** に短縮

このノートブックでは **MPS / CPU = fp32**, **CUDA = fp16 + `variant="fp16"`** で動かします。**MPS で fp32** を選ぶのは、上で述べた SDXL Base の MPS + fp16 NaN 問題を回避するための安全策です (SD1.5 の §2 で「fp32 + safety_checker ON」を選んだのと同じ趣旨)。

なお SDXL Base には **`safety_checker` が含まれません** (SD1.5 で同梱されていた CLIP ベースの NSFW 判定器は、SDXL 系では pipeline 既定構成に含まない方針に変わった)。

### model_id

In [ ]:
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
print("model_id:", MODEL_ID)

### Pipeline を読み込む

CUDA (fp16) では disk 節約のため `variant="fp16"` を指定し、MPS / CPU (fp32) では variant 指定なし (= fp32 既定 variant) で読み込みます。

In [ ]:
import time

from diffusers import StableDiffusionXLPipeline

# disk variant を dtype に合わせて選ぶ:
#   fp16 / bf16 → variant="fp16" (約 7 GB の fp16 重みを download)
#   fp32       → variant=None    (約 14 GB の fp32 既定 variant を download)
variant = "fp16" if dtype in (torch.float16, torch.bfloat16) else None

print(f"pipeline 読み込み中... variant={variant!r}")
print("(初回は約 7-14 GB の download を含むため十数分〜1 時間かかります)")
t_load = time.perf_counter()
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    use_safetensors=True,
    variant=variant,
)
pipe = pipe.to(device)
load_elapsed = time.perf_counter() - t_load

# MPS では attention 計算が重いので slicing を有効化する
if device == "mps":
    pipe.enable_attention_slicing()
    print("[mps] enable_attention_slicing() 有効化")

print(f"load done in {load_elapsed:.2f} s  (cache hit なら数秒、初回 download 時は十数分〜)")
print("pipeline クラス:", type(pipe).__name__)

<a id="gen"></a>

---
## 3. 画像生成

00 と同じ prompt / 同じ seed で 1 枚生成します。違いは:

- **解像度 512×512 → 1024×1024** (SDXL Base のネイティブ解像度)
- **steps は 30 が SDXL 既定** (SD1.5 の 40 から少し控えめ。SDXL は 1 step あたりの仕事量が大きい)
- **guidance_scale は 7.5 で同じ**

### 生成パラメータ、prompt、 seed を決める

In [ ]:
# 生成パラメータ (SDXL Base のネイティブ解像度)
width = 1024
height = 1024
num_inference_steps = 30       # SDXL 既定の典型値 (SD1.5 は 40 を使った)
guidance_scale = 7.5           # classifier-free guidance の強さ (00 と同じ)

In [ ]:
# 00 (SD1.5) と同じ「魔法使い」prompt / negative_prompt を使う。
# 同じ prompt + 同じ seed で SD1.5 と SDXL Base の出力差を直接比較するため。

prompt = (
    "a young witch with long flowing hair holding a glowing magic staff, "
    "in an enchanted forest with bioluminescent mushrooms, "
    "fantasy art, masterpiece, detailed"
)
negative_prompt = (
    "blurry, low quality, distorted, bad anatomy, extra limbs, "
    "watermark, signature"
)

# seed も 00 と同じ
seed = 123

In [ ]:
print("prompt          :", prompt)
print("negative_prompt :", negative_prompt)
print("seed            :", seed)
print("size            :", f"{width} x {height}")
print("steps           :", num_inference_steps)
print("guidance_scale  :", guidance_scale)

### 生成

`torch.Generator` で seed を固定するのは 00 と同じ。MPS では Generator を CPU で作る。

In [ ]:
import time
from pathlib import Path

# 生成画像の外部保存先 (notebook 内 cell 表示と同時に PNG として書き出す)
# notebook は lecture/ サブディレクトリから実行される前提 → 親ディレクトリ outputs/ を使う
NB_OUT_DIR = (Path("outputs/01_sdxl_base_intro") if IN_COLAB else Path("../outputs/01_sdxl_base_intro"))
NB_OUT_DIR.mkdir(parents=True, exist_ok=True)

generator = torch.Generator(device="cpu" if device == "mps" else device).manual_seed(seed)

print("生成中 ...")
t0 = time.perf_counter()
result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt or None,
    width=width,
    height=height,
    num_inference_steps=num_inference_steps,
    guidance_scale=guidance_scale,
    generator=generator,
)
elapsed = time.perf_counter() - t0
print(f"done in {elapsed:.2f} s")

image = result.images[0]  # pyright: ignore[reportAttributeAccessIssue]
out_path = NB_OUT_DIR / "sec3_base_30steps.png"
image.save(out_path)
print(f"saved: {out_path}")
image

**生成時間の感覚** (1024×1024 / 30 steps)

上のセルで実測されたのが、この環境での SDXL Base の生成時間です。

**M4 Max 実測リファレンス** (2026-05-24 / fp32 / MPS):

| notebook | モデル | 解像度 | steps | 生成時間 |
|---|---|---|---:|---:|
| 00 | SD1.5 | 512×512 | 40 | 9.95 s |
| **01 (このノート)** | **SDXL Base** | **1024×1024** | **30** | **50.87 s** |

つまり同じ M4 Max / MPS / fp32 で、**SDXL Base は SD1.5 の約 5.1 倍の時間**がかかります。

ざっくりの内訳としては:

- 解像度が縦横 2 倍 → 面積で **4 倍** の latent
- UNet パラメータが約 **3 倍** (860 M → 2.6 B)
- steps が 40 → 30 で **0.75 倍** (やや減る)

を掛け合わせると 4 × 3 × 0.75 = 9 倍 という素朴推算ですが、実測は 5.1 倍と推算より大幅に小さい。これは attention slicing / MPS の最適化具合・メモリ帯域律速など複合要因で、**1 step あたりの実コストは「params × 解像度面積」ほどには線形には伸びない**ことを示しています。

(00 で行った CPU 比較セクションを 01 では省略しています。SDXL Base を CPU で 1024×1024 を回すと M4 Max でも 10 分超かかる現実があるため。)

<a id="peek"></a>

---
## 4. パイプラインを覗いてみる

`pipe.components` の中身を見ます。00 (SD1.5) との差分:

- **text_encoder が 2 個** (`text_encoder`, `text_encoder_2`) / **tokenizer も 2 個** (`tokenizer`, `tokenizer_2`)
- **safety_checker / feature_extractor がない**

In [ ]:
# パイプラインの構成要素（component）名の一覧。
# 注意: 巨大な components dict をそのままセル出力にすると IPython が多重にキャッシュし
#       (Out/_N/DisplayHook 等)、§9 で解放してもメモリに残る。ここではキー名だけ表示する。
list(pipe.components.keys())

In [ ]:
import torch.nn as nn

# 生成パイプラインの中核モジュール (intro で「およそ 3.5B」と紹介したもの)
# SDXL では text_encoder が 2 個ある点が SD1.5 との大きな違い
core_modules = {"text_encoder", "text_encoder_2", "unet", "vae"}
core_total = 0
all_total = 0

for name, obj in pipe.components.items():
    cls = type(obj).__name__ if obj is not None else "None"
    if isinstance(obj, nn.Module):
        n_params = sum(p.numel() for p in obj.parameters())
        all_total += n_params
        if name in core_modules:
            core_total += n_params
        tag = "← 生成中核" if name in core_modules else ""
        print(f"  {name:20s} : {cls:32s}  {n_params/1e6:>7.1f} M  {tag}")
    else:
        print(f"  {name:20s} : {cls:32s}        -")

print()
print(f"  生成中核 (text_encoder × 2 + UNet + VAE) : {core_total/1e6:>7.1f} M  (≈ {core_total/1e9:.2f} B)")
print(f"  全 nn.Module 合計                       : {all_total/1e6:>7.1f} M")

主な役割 (SD1.5 との差分は **太字**):

| コンポーネント | 役割 |
|---|---|
| `tokenizer` | prompt → token id 列 (CLIP-L 用、SD1.5 と同じ vocab) |
| **`tokenizer_2`** | **prompt → token id 列 (OpenCLIP-bigG 用)** |
| `text_encoder` | token 列 → embedding (CLIP-L、SD1.5 と同じ) |
| **`text_encoder_2`** | **token 列 → embedding (OpenCLIP-bigG、より大きい)** |
| `unet` | latent から noise を予測。**SDXL では cross-attention の入力次元が 2048** |
| `vae` | pixel ↔ latent (SDXL 用に再学習されているが構造は SD1.5 と同じ AutoencoderKL) |
| `scheduler` | diffusion step を進める (SDXL の default は `EulerDiscreteScheduler`、SD1.5 は PNDM だった) |

**パラメータ規模**: 上の実測から、SDXL Base の中核は **text_encoder ≈ 123 M + text_encoder_2 ≈ 694 M + UNet ≈ 2.6 B + VAE ≈ 84 M = 約 3.5 B**。SD1.5 の 1.07 B と比べて **約 3.3 倍**になっています。UNet が依然として主役で全体の約 75% を占める構図は SD1.5 と同じ。

### 全体の流れ (SDXL の latent diffusion)

```
prompt (text)
  ↓ tokenizer  + text_encoder    → embed_L    (1, 77, 768)
  ↓ tokenizer_2 + text_encoder_2 → embed_bigG (1, 77, 1280) + pooled (1, 1280)
                ↓ concat
text embeddings (1, 77, 2048)   +  pooled embed (1, 1280)
  ↓ scheduler が denoising の時刻列と更新式を決める
  ↓ 各 step で UNet が noise を予測 → その予測に基づき latent を更新
clean latent (1, 4, 128, 128)
  ↓ VAE decoder                  # latent → pixel
画像 (3, 1024, 1024)
```

SD1.5 との骨格的な違いは **「text 側の embedding が 2 系統を concat した 2048 次元」** という一点。pooled embedding (OpenCLIP-bigG の文全体表現) も別途 UNet に渡されて、style 的な大局条件付けに使われます。

### モデルのバイト数を予測して disk cache と照合する

00 (SD1.5) と同じ手順。`variant=None` (fp32 既定) で download した場合の disk size を確認します。

`variant="fp16"` を指定した CUDA 環境では disk 上の重みが fp16 (2 bytes/param) なので `predicted / disk ≈ 2.0` (memory の方が disk の倍) になります。

In [ ]:
from pathlib import Path
from huggingface_hub import try_to_load_from_cache

WEIGHT_FILENAMES = (
    "diffusion_pytorch_model.safetensors",
    "model.safetensors",
    "diffusion_pytorch_model.bin",
    "model.bin",
)


def find_cache_size(repo_id: str, component: str, variant: str | None = None) -> int:
    # variant が指定されていれば、ファイル名に variant suffix が入った版を先に探す
    # (例: variant='fp16' なら 'model.fp16.safetensors')。
    if variant:
        for fname in WEIGHT_FILENAMES:
            stem, dot, ext = fname.rpartition(".")
            fname_v = f"{stem}.{variant}.{ext}" if dot else fname
            result = try_to_load_from_cache(repo_id=repo_id, filename=f"{component}/{fname_v}")
            if isinstance(result, str) and Path(result).exists():
                return Path(result).stat().st_size
    for fname in WEIGHT_FILENAMES:
        result = try_to_load_from_cache(repo_id=repo_id, filename=f"{component}/{fname}")
        if isinstance(result, str) and Path(result).exists():
            return Path(result).stat().st_size
    return 0


bytes_per_param = torch.empty(0, dtype=dtype).element_size()
print(f"current dtype : {dtype}  ({bytes_per_param} bytes/param)")
print(f"disk variant  : {variant!r}")
print(f"cache repo    : {MODEL_ID}\n")

print(f"  {'component':18s}  {'params':>8s}  {'predicted':>11s}  {'disk file':>11s}  {'ratio':>6s}")
print(f"  {'-'*18}  {'-'*8}  {'-'*11}  {'-'*11}  {'-'*6}")

pred_total = 0
disk_total = 0
for name, obj in pipe.components.items():
    if not isinstance(obj, nn.Module):
        continue
    n_params = sum(p.numel() for p in obj.parameters())
    predicted = n_params * bytes_per_param
    disk = find_cache_size(MODEL_ID, name, variant=variant)
    pred_total += predicted
    disk_total += disk
    ratio_str = f"{disk/predicted:.2f}x" if predicted > 0 and disk > 0 else "(n/a)"
    print(f"  {name:18s}  {n_params/1e6:>6.1f} M  {predicted/(1024**2):>8.1f} MB  {disk/(1024**2):>8.1f} MB  {ratio_str:>6s}")

print(f"  {'-'*18}  {'-'*8}  {'-'*11}  {'-'*11}  {'-'*6}")
total_ratio = f"{disk_total/pred_total:.2f}x" if pred_total > 0 else "(n/a)"
print(f"  {'total':18s}    {'':>6s}    {pred_total/(1024**3):>8.3f} GB  {disk_total/(1024**3):>8.3f} GB  {total_ratio:>6s}")

# snapshot dir 全体のサイズ
print()
sample = try_to_load_from_cache(repo_id=MODEL_ID, filename="model_index.json")
if isinstance(sample, str):
    snap_dir = Path(sample).parent
    cache_total = sum(p.stat().st_size for p in snap_dir.rglob("*") if p.is_file())
    others = cache_total - disk_total
    print(f"  snapshot dir 全体 (symlink 追跡、component 区別なし):")
    print(f"    cache 全体   : {cache_total/(1024**3):>7.3f} GB")
    print(f"    重み合計     : {disk_total/(1024**3):>7.3f} GB   (上の表の disk file 合計)")
    print(f"    その他       : {others/(1024**2):>7.2f} MB   (tokenizer × 2 / configs / scheduler / model_index.json 等)")
else:
    print("  (snapshot dir を解決できませんでした)")

### 観察: SD1.5 (00) との cache size 比較 — 実測値

00 §4 の表と並べて見ます (どちらも fp32 既定 variant、binary MB/GB):

| component | SD1.5 params | SD1.5 disk | SDXL params | SDXL disk |
|---|---:|---:|---:|---:|
| text_encoder | 123.1 M | 469.5 MB | 123.1 M | 469.5 MB (同じ CLIP-L) |
| text_encoder_2 | — | — | **694.7 M** | **2650 MB ≈ 2.59 GB** (OpenCLIP-bigG) |
| unet | 859.5 M | 3279 MB | **2567.5 M** | **9794 MB ≈ 9.56 GB** |
| vae | 83.7 M | 319.1 MB | 83.7 M | 319.1 MB |
| safety_checker | 304.0 M | 1160 MB | — | — (SDXL は含まない) |
| **重み合計** | **1.37 B** | **5.10 GB** | **3.47 B** | **12.92 GB** |

SDXL Base の重み合計は SD1.5 の **約 2.5 倍**。内訳としては UNet が 3.0 倍、新規 text_encoder_2 が 2.6 GB 加わったのが効いています。

### 観察: 「その他 = 322 MB」の内訳 — `vae_1_0/` という legacy ディレクトリ

clean な cache (新規 download 直後) で上の cell を実行すると:

```
cache 全体   :  13.237 GB
重み合計     :  12.923 GB   (上の表)
その他       :   322.17 MB
```

「その他 322 MB」の中身を実測で分解すると:

| ファイル | サイズ | 役割 |
|---|---:|---|
| `vae_1_0/diffusion_pytorch_model.safetensors` | **319 MB** | **使われない**重複 VAE (下で説明) |
| `tokenizer/vocab.json` + `tokenizer_2/vocab.json` | 計 ~2.1 MB | 2 つの CLIP tokenizer の vocab |
| `tokenizer/merges.txt` + `tokenizer_2/merges.txt` | 計 ~1.1 MB | BPE マージ規則 |
| 9 個の `*.json` (model_index, 各 component の config, scheduler config) | 計 ~6 KB | 構造定義 |

つまり「その他 322 MB」の **99% は `vae_1_0/` という 1 ディレクトリ**で、tokenizer や config は計 3 MB しか占めません。

### `vae_1_0/` とは — pipe からは使われない dead weight

`stabilityai/stable-diffusion-xl-base-1.0` の HF Hub repo には `vae/` と `vae_1_0/` の **2 つの VAE ディレクトリ**が並んでいます (2026-05 時点で WebFetch / `ls` 確認済)。

- `vae/`: 現用 VAE (pipeline が実際に使うもの)
- `vae_1_0/`: 元の v1.0 release 時の VAE を legacy 保存したもの

この実行環境の cache snapshot では `vae_1_0/` が含まれており、pipeline 構築時は `model_index.json` の `"vae"` エントリだけが参照されるため、`vae_1_0/` は `pipe.components` に登場しません (§4 の component 表で確認可能)。**少なくとも今回の実行では未使用の legacy VAE として cache に残った**ことが分かります。

cache の厳密な download 対象は diffusers / huggingface_hub のバージョンや `allow_patterns` 実装で変わり得るため、「必ず」とは断言しません。ただし `vae_1_0/` が含まれた場合は **~319 MB の使われない VAE が cache に残る** ことになり、これが SD1.5 (00 §4 で「その他 1.5 MB」) との差分の主因です。

### 注意: 過去に `variant="fp16"` を試したことがある場合の cache 肥大

このノートブックを実行する前に、もし**過去に同じ `MODEL_ID` を `variant="fp16"` で `from_pretrained()` した履歴があると、その時 download した fp16 variant ファイルが snapshot dir に同居して残ります**。HF cache は古い variant を自動削除しません。

例えば「最初 fp16 で試して MPS で NaN を踏み、fp32 に切り替えた」という探索パターンを踏んだ後の cache では、`*.fp16.safetensors` (4 components 分、約 6.4 GB) + `vae_1_0/*.fp16.safetensors` (167 MB) が snapshot 内に同居し、「その他」が **~7 GB** まで膨らみます。

このノートブックの table では `variant=None` で探索しているので、cohabit している fp16 variant のファイルは「重み合計」にカウントされず「その他」に流れ込みます。**「その他」が想定より明らかに大きい (~数 GB) ときは fp16 variant 残骸を疑うこと**。

clean 化するには:

```bash
# 該当 model の cache を完全削除 (副作用: 次回 from_pretrained で ~13 GB 再 download)
rm -rf ~/.cache/huggingface/hub/models--stabilityai--stable-diffusion-xl-base-1.0
```

部分削除 (fp16 ファイルだけ消したい) も可能ですが、HF cache は symlink + content-addressed blob 構造なので、blobs/ の該当ファイルと snapshots/ のシンボリックリンクの対応関係を辿る作業が必要になり、完全削除より手間がかかります。容量に余裕があれば cohabitation のまま放置しても動作上の害はありません。

### `variant` と `torch_dtype`: disk と memory の dtype は独立に指定する

`from_pretrained(model_id, variant=..., torch_dtype=...)` には dtype 関連のオプションが 2 つあり、**全く別の役割**を持ちます:

| オプション | 何を指定するか | いつ働くか |
|---|---|---|
| `variant=...` | **disk から download する重みファイルの dtype** (HF Hub 上の variant 選択) | 初回 download のとき (cache hit 後は何もしない) |
| `torch_dtype=...` | **メモリに load するときの dtype**。disk と違えば load 中に cast (型変換) が入る | `from_pretrained()` 呼び出しのたび |

SDXL Base には HF Hub 上に **fp32 (default) と fp16 (`variant="fp16"`)** の 2 variant が用意されています。組み合わせ:

| disk variant | `torch_dtype` | cast | disk | memory |
|---|---|---|---:|---:|
| default (= fp32) | `torch.float32` | なし | ~13 GB | ~13 GB |
| default (= fp32) | `torch.float16` | down-cast | ~13 GB | **~7 GB** |
| `variant="fp16"` | `torch.float32` | up-cast | **~7 GB** | ~13 GB |
| `variant="fp16"` | `torch.float16` | なし | ~7 GB | ~7 GB |

このノートブックでは MPS / CPU で「**(default, fp32)** = disk も memory も fp32」、CUDA で「**(fp16, fp16)** = disk も memory も fp16」を選んでいます (§1 で device 別 dtype を決め、§2 で variant をそれに揃える)。

### SDXL 固有: VAE fp16 NaN 問題

**一般事実 (HF community に広く知られている)**: SDXL の VAE は **fp16 decoding で数値オーバーフローを起こし NaN (= 黒画像) を出す**ことが知られている。証拠として有志製の差し替え VAE `madebyollin/sdxl-vae-fp16-fix` (fp16 で安定するよう修正された VAE) が広く使われている。

**この実行環境 (M4 Max + MPS) で観測した挙動**: MPS 上で fp16 / bf16 を使うと UNet / VAE 周辺で不安定性 (NaN を含む) を実測。**安全策として MPS では fp32 を使う**方針。

CUDA + fp16 で SDXL を動かす実用環境では、上記 NaN を回避する手段として以下のいずれかを採用するのが標準です:

1. **全部 fp32** で動かす (このノートブックの MPS で採用、安全だが disk/memory が重い)
2. UNet は fp16、**VAE だけ fp32 にキャスト** (`vae_fp32_override` パターン)
3. 有志製の差し替え VAE `madebyollin/sdxl-vae-fp16-fix` を使う (fp16 のまま安定)

なお (上の cache 肥大の節で触れた) **MPS で fp16 を試して NaN を踏んで fp32 に戻す** という探索を学生も自然に行いがちで、それが「**~7 GB の fp16 variant 残骸が cache に居座る**」原因になります。

<a id="inside"></a>

---
## 5. パイプラインの中身を見る

各コンポーネントを SD1.5 (00) と並べて観察します。

### text encoder 1 (CLIPTextModel) — SD1.5 と同じ CLIP ViT-L/14

SDXL の **1 つ目** の text encoder は SD1.5 と完全に同じ OpenAI CLIP ViT-L/14 の text 側です。

In [ ]:
text_encoder = pipe.text_encoder
text_cfg = text_encoder.config

print("class                :", type(text_encoder).__name__)
print("hidden_size          :", text_cfg.hidden_size)
print("num_hidden_layers    :", text_cfg.num_hidden_layers)
print("num_attention_heads  :", text_cfg.num_attention_heads)
print("max_position_embeddings:", text_cfg.max_position_embeddings)
print("vocab_size           :", text_cfg.vocab_size)

total = sum(p.numel() for p in text_encoder.parameters())
print(f"parameters           : {total / 1e6:.1f} M")

### text encoder 2 (CLIPTextModelWithProjection) — OpenCLIP ViT-bigG/14

SDXL の **2 つ目** の text encoder は LAION の OpenCLIP ViT-bigG/14 の text 側。SD1.5 にはなかったコンポーネントです。

- hidden_size **1280** (CLIP-L の 768 より大きい)
- num_hidden_layers **32** (CLIP-L の 12 より深い)
- token 列出力に加えて **pooled output** (文全体を 1 ベクトルに集約したもの) も返す

In [ ]:
text_encoder_2 = pipe.text_encoder_2
text2_cfg = text_encoder_2.config

print("class                :", type(text_encoder_2).__name__)
print("hidden_size          :", text2_cfg.hidden_size)
print("num_hidden_layers    :", text2_cfg.num_hidden_layers)
print("num_attention_heads  :", text2_cfg.num_attention_heads)
print("max_position_embeddings:", text2_cfg.max_position_embeddings)
print("vocab_size           :", text2_cfg.vocab_size)
print("projection_dim       :", text2_cfg.projection_dim, " (pooled output dim)")

total = sum(p.numel() for p in text_encoder_2.parameters())
print(f"parameters           : {total / 1e6:.1f} M")

**2 つの text encoder の関係**

SDXL の text 条件付けは、**両方の encoder の出力を concat** して 1 本の embedding にしてから UNet に渡します:

```
embed_L    : (1, 77, 768)   ← text_encoder    (CLIP-L)
embed_bigG : (1, 77, 1280)  ← text_encoder_2  (OpenCLIP-bigG)
              ↓ 最後の次元で concat
text_emb   : (1, 77, 2048)  ← UNet の cross-attention 入力
```

これが SDXL UNet の `cross_attention_dim=2048` の由来です (pooled output や time embedding 経由の渡し方を含む全体像は [§6 末尾の Remark](#tokenizer) で整理)。

### UNet (UNet2DConditionModel) — SDXL では約 3 倍重い

In [ ]:
unet = pipe.unet
unet_cfg = unet.config

print("class                :", type(unet).__name__)
print("sample_size          :", unet_cfg.sample_size, " (latent edge length)")
print("in_channels          :", unet_cfg.in_channels,  " (latent channel)")
print("out_channels         :", unet_cfg.out_channels, " (predicted noise channel)")
print("cross_attention_dim  :", unet_cfg.cross_attention_dim, " (= 768 + 1280 = concat した text emb の dim)")
print("down_block_types     :", unet_cfg.down_block_types)
print("up_block_types       :", unet_cfg.up_block_types)
print("block_out_channels   :", unet_cfg.block_out_channels)
print("addition_embed_type  :", getattr(unet_cfg, "addition_embed_type", None), " (SDXL の追加条件付け)")

total = sum(p.numel() for p in unet.parameters())
print(f"parameters           : {total / 1e6:.1f} M")

**UNet の特徴 — SD1.5 との差分**

| 項目 | SD1.5 | SDXL Base |
|---|---|---|
| `sample_size` | 64 (= 512 / 8) | **128** (= 1024 / 8) |
| `in_channels` / `out_channels` | 4 / 4 | 4 / 4 (同じ) |
| `cross_attention_dim` | 768 | **2048** (= 768 + 1280 を concat) |
| `block_out_channels` | [320, 640, 1280, 1280] | **[320, 640, 1280]** (3 段、SD1.5 は 4 段だった) |
| 段数 | 4 段 down + 4 段 up | **3 段 down + 3 段 up** (より浅いが各段が太い) |
| `addition_embed_type` | `None` | **`"text_time"`** (pooled emb + 解像度・crop 条件) |
| パラメータ | 約 860 M | **約 2.6 B** (約 3 倍) |

`addition_embed_type="text_time"` は SDXL 固有で、OpenCLIP-bigG の pooled embedding と「target_size / original_size / crop_top_left」という micro-conditioning 情報を UNet に追加で渡す仕組みです (論文 SDXL §2.2 参照)。

### Remark — time embedding とは

`addition_embed_type="text_time"` を理解するには **time embedding** の概念が前提:

**time embedding = diffusion step `t` を全層に共有するための仕組み**

- scheduler が出す `t` (整数 0..1000) を **sinusoidal + MLP** で 1280 次元 vector に変換
- UNet 内の各 ResNet block の feature map に **add 注入** (空間方向 broadcast)
- 1 つの UNet が 30〜1000 段階の denoising を兼任するため、t を全層で共有する必要がある
- LLM の `position embedding` が「token 列のどこか」を伝えるのと analogous で、**diffusion の step 軸版**

**SDXL の `text_time` 拡張**: time embedding channel は **per-image な global broadcast 入口** なので、t 以外の **画像全体に効く global 条件** も同居可能。SDXL はこれを活かして:

| 追加項目 | 次元 | 内容 |
|---|---|---|
| `pooled` (text_embeds) | (1, 1280) | OpenCLIP-bigG の pooled output、文全体の global text 表現 |
| `time_ids` | (1, 6) | `[orig_h, orig_w, crop_top, crop_left, target_h, target_w]` micro-conditioning |

を time embedding に **add 混合**。これが `addition_embed_type="text_time"` の中身。

**cross-attention vs time embedding の役割分担**:

| 経路 | 性質 | 入る情報 |
|---|---|---|
| cross-attention の K, V | **per-token**: 空間位置 × token 対応 | `text_emb` (1, 77, 2048) = 2 encoder concat 後 |
| time embedding channel | **per-image**: global broadcast (空間方向均等) | `t` + `pooled` + `time_ids` |

「**空間依存条件** (画像のどこに何を置くか) は cross-attention、**画像全体条件** (時刻・スタイル全体・解像度) は time embedding」という構造的分業。Diffusers の `addition_embed_type` 設定 (`None` / `text_time` / `text` / `image` 等) は **time channel = 拡張可能な global conditioning slot** という設計を反映している。


### VAE (AutoencoderKL) — 構造は SD1.5 と同じ、SDXL 用に再学習

In [ ]:
vae = pipe.vae
vae_cfg = vae.config

print("class              :", type(vae).__name__)
print("in_channels        :", vae_cfg.in_channels,  " (RGB)")
print("out_channels       :", vae_cfg.out_channels, " (RGB)")
print("latent_channels    :", vae_cfg.latent_channels)
print("block_out_channels :", vae_cfg.block_out_channels)
print("scaling_factor     :", vae_cfg.scaling_factor)

total = sum(p.numel() for p in vae.parameters())
print(f"parameters         : {total / 1e6:.1f} M")

**VAE の特徴**

- 構造は SD1.5 と同じ AutoencoderKL、空間 1 辺を 1/8 に圧縮、channel 3 → 4
- 1024×1024 入力 → 128×128 latent (SD1.5 の 512 → 64 と同じ「8 倍」比率)
- **`scaling_factor` が SD1.5 (0.18215) と少し違う** ことがある (SDXL では 0.13025、再学習で latent 分散が変わったため。上の出力で確認)
- パラメータ数 83.7 M は SD1.5 と同じ — 構造は変えず、SDXL の高解像度学習データで重みを再フィットしたもの

### scheduler — SDXL 既定は EulerDiscreteScheduler

SD1.5 の default が PNDMScheduler だったのに対し、SDXL の default は `EulerDiscreteScheduler` (一般に「Euler」と呼ばれるもの)。

In [ ]:
scheduler = pipe.scheduler

print("class                :", type(scheduler).__name__)
print("num_train_timesteps  :", scheduler.config.num_train_timesteps)
print("beta_schedule        :", scheduler.config.beta_schedule)
print("prediction_type      :", scheduler.config.prediction_type)

**scheduler の差分**

- SD1.5 の default = `PNDMScheduler` (2022 年公開時の選択)
- SDXL の default = `EulerDiscreteScheduler` (現代的な低 step 高品質スケジューラ)
- どちらも別の scheduler (DPMSolver, DDIM 等) に差し替え可能。trade-off (品質 ↔ 速度) を notebook 単位で実験する遊び方は SD1.5 と同じ


<a id="tokenizer"></a>

---
## 6. tokenizer の動作を確認 — 2 つの tokenizer の対比

SDXL は tokenizer が 2 つあります (`tokenizer` と `tokenizer_2`)。一般論としては別 tokenizer なら vocab も BPE 規則も独立ですが、**SDXL Base 1.0 公式 repo では両 tokenizer に同じ `CLIPTokenizer` ファイル (同じ vocab + 同じ BPE 規則) が置かれており、実質同じものが 2 つある状態**です。「tokenizer 2 つ」は token 化レイヤの差ではなく、後段で **同じ token id 列を 2 つの異なる text encoder に流す入口** にあたります。

ここでは 00 と同じ prompt を両方の tokenizer に通し、**content 部分の** token id 列が完全一致することを確認します（通常語彙・BPE は同一。padding 等の special token だけは pad_token_id が異なる＝後述「微差」）。

In [ ]:
tokenizer = pipe.tokenizer
tokenizer_2 = pipe.tokenizer_2

ids_1 = tokenizer.encode(prompt)
ids_2 = tokenizer_2.encode(prompt)

print(f"prompt    : {prompt}")
print(f"tokenizer   (CLIP-L)        : {len(ids_1)} tokens")
print(f"tokenizer_2 (OpenCLIP-bigG) : {len(ids_2)} tokens")
print()
print(f"  {'idx':>3s}  {'id_L':>6s}  {'piece_L':25s}  {'id_bigG':>7s}  {'piece_bigG':25s}  same?")
print(f"  {'-'*3}  {'-'*6}  {'-'*25}  {'-'*7}  {'-'*25}  -----")
for i in range(max(len(ids_1), len(ids_2))):
    t1 = ids_1[i] if i < len(ids_1) else None
    t2 = ids_2[i] if i < len(ids_2) else None
    p1 = tokenizer.decode([t1]) if t1 is not None else ""
    p2 = tokenizer_2.decode([t2]) if t2 is not None else ""
    same = "✓" if t1 == t2 else "✗"
    t1s = f"{t1:6d}" if t1 is not None else "    - "
    t2s = f"{t2:7d}" if t2 is not None else "     - "
    print(f"  [{i:2d}]  {t1s}  {p1!r:25s}  {t2s}  {p2!r:25s}  {same}")

# padded encoding で pad 位置の差を確認 (SDXL pipeline 内部での実形式)
tokens_L_padded = tokenizer(prompt, padding="max_length",
                            max_length=tokenizer.model_max_length,
                            truncation=True, return_tensors="pt")
tokens_G_padded = tokenizer_2(prompt, padding="max_length",
                              max_length=tokenizer_2.model_max_length,
                              truncation=True, return_tensors="pt")
ids_L_full = tokens_L_padded.input_ids[0].tolist()
ids_G_full = tokens_G_padded.input_ids[0].tolist()

print()
print("=== padded encoding (max_length=77、SDXL pipeline 内部での実形式) ===")
print(f"  total length: CLIP-L={len(ids_L_full)}, OpenCLIP-G={len(ids_G_full)}")
print(f"  full 77 ids 完全一致?: {ids_L_full == ids_G_full}")
print()
print(f"  content + BOS + EOS (positions 0..{len(ids_1)-1}) 一致?: {ids_L_full[:len(ids_1)] == ids_G_full[:len(ids_2)] == ids_1}")
print()
print(f"  pad 区間 (positions {len(ids_1)}..76、計 {77 - len(ids_1)} 位置):")
print(f"    CLIP-L     pad ids : {ids_L_full[len(ids_1):]}")
print(f"    OpenCLIP-G pad ids : {ids_G_full[len(ids_2):]}")
print(f"    pad 区間 完全一致?  : {ids_L_full[len(ids_1):] == ids_G_full[len(ids_2):]}")


**観察ポイント**

- 同じ CLIP BPE 系列なので **token 列の長さは一致** (今回の prompt では 33 token / 33 token)
- 上の表の `same?` 列を見ると、**全 33 token の id が完全一致**している。これは SDXL Base 1.0 の HF repo では `tokenizer` と `tokenizer_2` がどちらも同じ `CLIPTokenizer` ファイル (同じ vocab.json / merges.txt) を使っているため
- つまり SDXL の「tokenizer が 2 つある」は **token 化レイヤの違いではない**。**同じ token id 列を、2 つの異なる text encoder (CLIP-L 768 次元 / OpenCLIP-bigG 1280 次元) に通すことで 2 系統の embedding が得られる**点が本質
- (理論上は他の SDXL 派生モデルが tokenizer_2 を別 vocab で配布する可能性はあるが、SDXL Base 1.0 公式 repo では同じ tokenizer を 2 重に持つ構成)
- どちらの tokenizer も最大長 77 / `<|startoftext|>` `<|endoftext|>` で囲む CLIP の慣習は共通
- **微差**: `pad_token_id` のみ異なる (`tokenizer` = 49,407 = EOS と同じ id を流用、`tokenizer_2` = 0)。上の demo 後半で padded encoding (`padding="max_length", max_length=77`、SDXL pipeline 内部での実形式) を確認できる。pad 区間で 2 系統の id が異なる:
  - `tokenizer` (CLIP-L) padded: `[BOS, content..., EOS, 49407, 49407, ...]` ← pad = EOS id なので **EOS と pad の区別がつかない**
  - `tokenizer_2` (OpenCLIP-bigG) padded: `[BOS, content..., EOS, 0, 0, ...]` ← EOS (49407) と pad (0) が区別可能
  - 実 padded encoding の出力比較は [02ノートブック §4](02_sdxl_base_inside.ipynb#tokenize) で確認できる (両 tokenizer に `padding="max_length"` を渡して全 77 要素を表示)。content / BOS / EOS は両 tokenizer で完全一致なので、生成挙動への影響は実用上ほぼなし

---

### Remark — SDXL の text 条件付け 全体像 (concat も含めて)

01 全体で **concat** は §0 の SDXL Base 表、§5 の concat 図、§5 末尾の比較表、上の §6 観察と複数箇所に断片的に登場するので、ここで全体像を一度整理:

```
[1] prompt → 2 種の token 列
    prompt → tokenizer    → ids_L (1, 77)
    prompt → tokenizer_2  → ids_G (1, 77)
    (SDXL Base 1.0 では同 vocab → content / BOS / EOS の id は ids_L = ids_G)

[2] 2 つの text encoder に通す
    ids_L → text_encoder    → embed_L  (1, 77, 768)
    ids_G → text_encoder_2  → embed_G  (1, 77, 1280)
                              pooled_G (1, 1280)        ← bigG だけが返す

[3] cross-attention 用: embed を concat
    embed_L (768) + embed_G (1280)
      → torch.cat(dim=-1)
      → text_emb (1, 77, 2048)
      → UNet の cross-attention の K, V

[4] time embedding 用: pooled を注入
    pooled_G (1, 1280)
      → time embedding に注入
      → UNet の time 条件 (大局スタイル + 解像度・crop)
```

ポイント:

- **concat する次元**: tensor の最終 axis (channel/feature 軸)。token 軸 (77) では concat しない
- **token 軸の整合性**: padding で 77 に揃えるので 2 系統とも `(1, 77, ...)` で長さ一致、最終次元方向 (768, 1280) を結合して 2048 になる
- **`cross_attention_dim=2048`** (§5 の UNet config 表示で実測): UNet 側がこの concat 後 dim を K, V として受け取る
- **pooled output**: text_encoder_2 だけが pooled (1, 1280) を返す。これは concat には参加せず、**time embedding 経由**で UNet に渡る (`addition_embed_type="text_time"`、§5 末尾の比較表で言及)
- **`tokenizer` と `tokenizer_2` の vocab が一致** している (SDXL Base 1.0 公式) のは、token 化レイヤの違いではなく **encoder の違いだけが本質** という重要観察 (上の bullet と同じ点)

01 までは各 component の config (768d + 1280d encoder + 2048d UNet) を観察し、**concat → 2048** の構造を逆算で確認しました。実際に concat 操作を tensor の shape まで実演する手続きは [02 ノートブック §4](02_sdxl_base_inside.ipynb#tokenize) で行います。



<a id="sweep"></a>

---
## 7. seed と steps を変えてみる

00 と全く同じ実験を SDXL でも繰り返します。

### seed を変える

In [ ]:
def generate(
    *,
    prompt: str,
    negative_prompt: str | None = None,
    seed: int,
    num_inference_steps: int,
    guidance_scale: float = 7.5,
):
    # 00 と同じ小さなラッパー。SDXL Base 用 (1024×1024)。
    # §3 の base 画像と公平比較するため negative_prompt を必ず渡す。
    g = torch.Generator(device="cpu" if device == "mps" else device).manual_seed(seed)
    out = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt or None,
        width=1024,
        height=1024,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=g,
    )
    return out.images[0]  # pyright: ignore[reportAttributeAccessIssue]


img = generate(
    prompt=prompt,
    negative_prompt=negative_prompt,
    seed=seed + 100,
    num_inference_steps=num_inference_steps,
)
out_path = NB_OUT_DIR / "sec7_seed_plus100.png"
img.save(out_path)
print(f"saved: {out_path}")
img

### steps を変える

SDXL でも 00 と同じく `num_inference_steps` を `// 2` / `// 4` に減らして比較します。SDXL は SD1.5 より低 step 耐性がやや高い (Euler scheduler の効果) と言われていますが、確かめてみましょう。

In [ ]:
img = generate(
    prompt=prompt,
    negative_prompt=negative_prompt,
    seed=seed,
    num_inference_steps=num_inference_steps // 2,
)
out_path = NB_OUT_DIR / "sec7_steps_half.png"
img.save(out_path)
print(f"saved: {out_path}")
img

In [ ]:
img = generate(
    prompt=prompt,
    negative_prompt=negative_prompt,
    seed=seed,
    num_inference_steps=num_inference_steps // 4,
)
out_path = NB_OUT_DIR / "sec7_steps_quarter.png"
img.save(out_path)
print(f"saved: {out_path}")
img

**比較**

§3 の base `num_inference_steps` (現在 30) を基準に:

- **base = 30 steps** (§3 で生成済): SDXL Base の標準的な品質
- **`// 2 = 15` steps**: SDXL は 15 steps 程度ならまだ実用範囲。Euler scheduler が低 step に強い
- **`// 4 = 7` steps**: ノイズ感が残る。base モデルではここまで下げると厳しい

**SD1.5 と比べた感覚**:

| | SD1.5 (00) | SDXL Base (01) |
|---|---|---|
| 実用下限 steps | 20 程度 | **15 程度** (Euler) |
| 標準 steps | 30〜50 | **25〜40** |
| 上限 (飽和) | 50 以上は飽和 | **同様に飽和** |

次の §8 で見る SDXL Turbo は、これをさらに極端にして「**1〜4 steps で動く**」ように蒸留 (distillation) したモデルです。

<a id="lightning"></a>

---
## 8. efficiency LoRA: SDXL Lightning (蒸留版、Turbo の進化形)

00 では SD1.5 の **FT (fine-tune) 派生** モデル (anime FT 等) と比較しました。01 では、同じ SDXL Base から作られた、より構造的な派生である **SDXL Lightning** を試します (HF Hub レポジトリ [`ByteDance/SDXL-Lightning`](https://huggingface.co/ByteDance/SDXL-Lightning))。

### SDXL Lightning とは

2024 年に ByteDance がリリースした SDXL Base の **distillation (蒸留)** モデル。先発の SDXL Turbo (Stability AI, 2023 年) の後継世代で、Turbo より高品質と評価されています。

- **アーキテクチャは SDXL Base と完全に同じ** (text encoder × 2、UNet、VAE すべて同じ shape)
- **蒸留学習の対象は UNet のみ**。text encoder × 2 と VAE は **SDXL Base のものをそのまま流用**している (これは Lightning だけでなく Turbo / Hyper-SDXL 等の step 削減系蒸留に共通する設計上の事実 — 蒸留は「step 数を削る」が目的で、text encoder / VAE は denoising loop の外なので触る必要がない)
- **1 / 2 / 4 / 8 step 蒸留版** が用意されている (このノートブックでは 4 step 版を使う)
- guidance_scale **= 0.0** (Turbo と同じく CFG なしで動くよう蒸留)
- **scheduler を Euler + `timestep_spacing="trailing"` に設定する必要がある** (Lightning 固有のお作法。これを忘れるとぼやけた画像になる)
- **1024×1024 で安定** (Turbo は 512×512 中心の蒸留だったが、Lightning は SDXL Base のネイティブ解像度をそのまま継承)

### Lightning の 3 つの配布形式 (HF Hub [`ByteDance/SDXL-Lightning`](https://huggingface.co/ByteDance/SDXL-Lightning) repo)

ByteDance は同じ蒸留モデルを **3 つのファイル形式**で並行配布しています。4 step 版の場合:

| 形式 | ファイル名 (4-step) | サイズ | 中身 | Diffusers での読み方 |
|---|---|---:|---|---|
| **full single-file** | `sdxl_lightning_4step.safetensors` | 6.94 GB | UNet + text encoder × 2 + VAE 全部 (fp16) を 1 ファイルに詰めた AUTOMATIC1111/stable-diffusion-webui / ComfyUI 系形式 | `StableDiffusionXLPipeline.from_single_file(...)` |
| **UNet only** | `sdxl_lightning_4step_unet.safetensors` | 5.14 GB | UNet 重みだけ (fp16) | Base から空 UNet 作成 + `load_state_dict` |
| **LoRA delta** | `sdxl_lightning_4step_lora.safetensors` | **394 MB** | UNet を LoRA で書き換える差分 | `pipe.load_lora_weights(...)` + `pipe.fuse_lora()` |

(上記ファイル一覧・サイズは 2026-05 時点で HF Hub 上にて確認。すべて公式の `ByteDance/SDXL-Lightning` repo にあり、ユーザは目的に応じて選べる)

### 公式の品質推奨: Full UNet > LoRA

**重要 (model card 引用)**: ByteDance は model card で品質順序を明示しています。

> "We provide both full UNet and LoRA checkpoints. The **full UNet models have the best quality** while the LoRA models can be applied to other base models."
> "**Use LoRA only if you are using non-SDXL base models. Otherwise use our UNet checkpoint for better quality.**"

つまり 3 形式の関係は:

- **full single-file / UNet only**: どちらも完全な Lightning UNet 重みを含むため、出力品質は同等 (実質同じ重みで包装フォーマットが違うだけ)
- **LoRA delta**: 「Base UNet → Lightning UNet」の差分を **低ランク近似 (rank ~64〜128 程度)** で表したもの。Lightning UNet の**近似**であり、Full UNet checkpoint より品質は **わずかに劣る** ことが公式に示唆されている
- **公式は SDXL Base に対しては Full UNet checkpoint を推奨**、LoRA は「**他の base model (例えば Animagine XL) に Lightning 化を適用したいとき**」の用途に位置付けている

### このノートブックで採用する方式と理由: **LoRA delta** (教育目的)

公式は Full UNet を推奨していますが、このノートブックでは **教育目的を優先**して LoRA delta を採用します:

- **追加 download が ~394 MB だけで済む** (Full の 6.94 GB / UNet only の 5.14 GB と比べて **~13×〜18× 小さい**)。「Base 共有による節約」の story が成立する唯一の形式
- **LoRA は現代 diffusion エコシステムの基本概念**: SD1.5 / SDXL の FT 派生 (anime / photoreal / style) は LoRA 配布が主流、ControlNet との組み合わせや multi-LoRA stacking 等の後続トピックの基礎になる。講義で導入する価値が高い
- **コードが LoRA API として直感的**: `pipe.load_lora_weights(repo, weight_name=...)` + `pipe.fuse_lora()` の 2 行で完了 (UNet only の `load_config + from_config + load_state_dict + 新 pipeline 構築` という低レベル手作業より読みやすい)
- 「蒸留の正体 = Base UNet との **小さな低ランク差分**」が**量で**示せる: たった 394 MB の delta で Lightning 化できる事実は、「蒸留は UNet 全体を作り直しているのではなく、軽い修正で step 数だけを大きく削っている」という本質と整合する
- 一方、**最高品質を求めるなら Full UNet checkpoint** を使うべきという公式 stance は §10 の Animagine と組み合わせるときに改めて関連してくる

### LoRA とは (簡単な背景)

**LoRA (Low-Rank Adaptation)** は、巨大な行列 W に追加学習を入れる時に、その差分 ΔW を 2 つの低ランク行列の積 ΔW = B·A で近似する手法。元の W (~2.6 B params) と比べて、A と B は計 ~数十〜数百 M params 程度で済むため **非常に軽量**。生成時には ΔW = B·A を base に加算 (または fuse) して使う。

オリジナル論文は LLM 用 (Hu et al., 2021, arXiv:2106.09685)。diffusion での流行は 2023 年以降の SD1.5 FT エコシステムから。Lightning のような distillation でも LoRA 表現を採用することで、配布サイズを劇的に圧縮できる。

### このノートでの実装方針

§3 の `pipe` (SDXL Base) を **温存**したいので、新しい `pipe_alt` を作って:

1. **Base の text encoder × 2 / VAE / tokenizer × 2 はリファレンス共有** (メモリ追加なし)
2. **UNet だけ `copy.deepcopy()`** して `pipe_alt` に持たせる (Base UNet と別インスタンスにする → LoRA fuse の副作用が §3 pipe に波及しない)
3. scheduler は新規に Euler + trailing で作る
4. `pipe_alt.load_lora_weights(...)` + `pipe_alt.fuse_lora()` で LoRA を pipe_alt.unet に焼き込む

これで §3 と §8 を独立に並べて見られます (追加メモリは UNet クローン分の ~10 GB のみ)。

In [ ]:
import copy
from diffusers import EulerDiscreteScheduler

LIGHTNING_REPO = "ByteDance/SDXL-Lightning"
LIGHTNING_LORA_CKPT = "sdxl_lightning_4step_lora.safetensors"   # 4-step 蒸留版の LoRA delta (~394 MB)

print(f"loading LoRA: {LIGHTNING_REPO} / {LIGHTNING_LORA_CKPT}")
print("(初回 download は ~394 MB — Lightning UNet 5.14 GB の ~1/13)")
t_load_alt = time.perf_counter()

# Step 1: pipe_alt を組み立てる
# - Base の text encoder × 2 / VAE / tokenizer × 2 は §3 pipe と「同じインスタンス」を共有 (リファレンス渡し、メモリ追加なし)
# - UNet だけ deepcopy して「§3 pipe の UNet とは別物」にする → LoRA fuse の副作用が §3 pipe.unet に波及しない
# - scheduler は新規に Lightning 用 Euler + trailing spacing を作る
unet_alt = copy.deepcopy(pipe.unet)
scheduler_alt = EulerDiscreteScheduler.from_config(
    pipe.scheduler.config, timestep_spacing="trailing"
)
pipe_alt = StableDiffusionXLPipeline(
    vae=pipe.vae,
    text_encoder=pipe.text_encoder,
    text_encoder_2=pipe.text_encoder_2,
    tokenizer=pipe.tokenizer,
    tokenizer_2=pipe.tokenizer_2,
    unet=unet_alt,
    scheduler=scheduler_alt,
)
pipe_alt = pipe_alt.to(device)

if device == "mps":
    pipe_alt.enable_attention_slicing()

# Step 2: Lightning LoRA を pipe_alt.unet に load + fuse
# - load_lora_weights: HF Hub から LoRA delta tensor を download し、UNet の対応する layer に紐付け
# - fuse_lora: LoRA delta (B·A) を base UNet 重みに「焼き込む」 (生成時の計算が軽くなる、後で unfuse_lora で戻せる)
pipe_alt.load_lora_weights(LIGHTNING_REPO, weight_name=LIGHTNING_LORA_CKPT)
pipe_alt.fuse_lora()

load_elapsed_alt = time.perf_counter() - t_load_alt
print(f"load done in {load_elapsed_alt:.2f} s")
print("pipeline クラス :", type(pipe_alt).__name__)
print("scheduler       :", type(pipe_alt.scheduler).__name__, "(timestep_spacing=trailing)")
print("UNet            : Base UNet (deepcopy) + Lightning LoRA fused")

In [ ]:
# Lightning 用の生成パラメータ (LoRA fused でも Lightning 本来と同じパラメータで生成する)
# - num_inference_steps は使う LoRA checkpoint の step 数に合わせる (4-step 版なので 4)
# - guidance_scale = 0.0 (Turbo と同じく CFG なしで動くよう蒸留)
# - 解像度は SDXL Base と同じ 1024×1024 (Lightning は 1024 で安定、Turbo との大きな違い)
alt_steps = 4
alt_guidance = 0.0
alt_width = 1024
alt_height = 1024

# §3 で作った generator を再 seed して使い回す (00 と同じパターン)
generator.manual_seed(seed)

print(f"生成中 with {LIGHTNING_REPO} (LoRA fused, steps={alt_steps}, guidance={alt_guidance}, size={alt_width}x{alt_height}) ...")
t0 = time.perf_counter()
result_alt = pipe_alt(
    prompt=prompt,
    width=alt_width,
    height=alt_height,
    num_inference_steps=alt_steps,
    guidance_scale=alt_guidance,
    generator=generator,
)
elapsed_alt = time.perf_counter() - t0
print(f"done in {elapsed_alt:.2f} s   (SDXL Base 30 steps = {elapsed:.2f} s の何倍速?)")

image_alt = result_alt.images[0]  # pyright: ignore[reportAttributeAccessIssue]
out_path = NB_OUT_DIR / "sec8_lightning_4steps.png"
image_alt.save(out_path)
print(f"saved: {out_path}")
image_alt

**比較**

| | SDXL Base (§3) | SDXL Lightning (§8) |
|---|---|---|
| 重み出自 | Stability AI が直接学習 | ByteDance が SDXL Base を蒸留 (2024) |
| 配布形態 (このノートで使ったもの) | 完結 pipeline (text encoder × 2 + UNet + VAE) | **LoRA delta** (394 MB ≈ 376 MiB) を Base UNet に適用 |
| アーキテクチャ | — | **同じ** (text encoder × 2 + UNet + VAE、shape も同一) |
| 標準 steps | 30 | **4** (1 / 2 / 4 / 8 step LoRA あり) |
| guidance_scale | 7.5 | **0.0** (CFG なし) |
| scheduler | EulerDiscreteScheduler (既定) | **EulerDiscreteScheduler + timestep_spacing="trailing"** |
| 推奨解像度 | 1024×1024 | **1024×1024** (Lightning は Base の解像度をそのまま継承) |
| **追加 download (Base が cache 済の場合)** | — | **~376 MiB** (LoRA delta だけ) |
| **生成時間 (M4 Max 実測)** | **50.87 s** (30 steps) | **4.88 s** (4 steps) → **base の約 1/10.4** |

### サイズの読み方注意: 「394 MB」と「376 MiB」は同じ値

LoRA ファイルの公称サイズには 2 種類の数字が登場します:

- **394 MB** (HF Hub の表示) — decimal 単位、1 MB = 10⁶ bytes
- **376 MiB** (このノート §4 の cache size 計算と同じ binary 単位、1 MiB = 2²⁰ bytes)

どちらも同じファイルを指しています (`394 × 10⁶ ≈ 376 × 2²⁰`)。混乱するので、§4 の cache 表は binary、HF Hub 表示は decimal、と覚えておくと良いです。

### LoRA params 量で見る「delta の軽さ」

実測 (`safetensors.torch.load_file()` で LoRA を読んで params を数えると):

- **LoRA params 合計: 196.75 M** (788 layer × 3 = alpha + lora_down + lora_up = 2364 tensor)
- これは **Base UNet の 2.6 B params の ~7.5%**
- 言い換えると: Base UNet 全体を作り直さず、**~7.5% 程度の追加 params** だけで「30 step → 4 step 高速化」が実現できている

**観察ポイント**

- **steps が 30 → 4 に減っているのに、解像度は base と同じ 1024**。同じ画面サイズで時間だけ短くなっているので、Base ↔ Lightning の品質差が直接見える
- 体感としては **Turbo よりは明確に良い**が、**Base 30 step ほどの品質には届かない**ことが多い (色味・細部・破綻のなさで Base に分がある)。これは「蒸留は元モデルの完全再現ではなく、速度との trade-off」という基本原則そのもの。**公式は最高品質には Full UNet checkpoint を推奨**しており、LoRA delta は「他の base model に Lightning 化を適用したいとき」の用途、というのが ByteDance の position
- **LoRA 配布のサイズ感の衝撃**: Base UNet が 9.79 GB あるのに対して、Lightning 化に必要な delta はわずか **~376 MiB** (~7.5% の追加 params)。UNet 全体を作り直す必要は無く、低ランク (~64-128 rank) の小さな修正だけで 30 step → 4 step を実現できる。蒸留の本質が「重みの大改造」ではなく「step 数を減らすためのささやかな調整」だと量的に分かる
- 速度比 ~10× は **steps の比 30/4 = 7.5×** とほぼ一致 (実測 10.4× はやや上回る)。これは「**1 step あたりの計算量は Base と Lightning で同じ** (UNet サイズが同一、LoRA は fuse 済で実行時オーバーヘッドなし)、節約しているのは step 数だけ」という事実をそのまま反映している。+ MPS の overhead が刻みごとに乗るため、step 数を減らした効果は理論値より少し有利に出ることがある
- **`load_lora_weights()` が出す warning は、この LoRA file が UNet 側だけを変更している証拠**: cell [49] 実行時に diffusers が以下の warning を出す:
  ```
  No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'.
  No LoRA keys associated to CLIPTextModelWithProjection found with the prefix='text_encoder_2'.
  ```
  これは「この LoRA file には text_encoder / text_encoder_2 用の LoRA key が含まれていない」という事実そのもの。すなわち**今回読み込んだ Lightning LoRA file が変更しているのは UNet 側だけ**で、text encoder には触らない。ただし **Diffusers の `load_lora_weights()` API 自体は text encoder 用 LoRA も扱える**ので、「全 SDXL LoRA は UNet だけ」と一般化するのは誤り。あくまで「**この checkpoint の場合**」の話、と理解する
- guidance_scale を変えると出力は変わるが、**公式推奨の `guidance_scale=0.0` から外れると品質が崩れやすい** (Lightning は CFG なしで動くよう蒸留されているため、CFG を勝手に足すと蒸留学習の前提から外れる)
- scheduler の `timestep_spacing="trailing"` を **忘れるとぼやけた画像になる**。Lightning 固有のお作法
- 品質をもう一段上げたいときは 8 step 版 LoRA (`sdxl_lightning_8step_lora.safetensors`) に差し替える手がある。step が倍になる代わりに Base により近づく。**最高品質を求めるなら LoRA ではなく Full UNet checkpoint** を使うのが公式推奨ルート
- (歴史) Turbo (Stability AI, 2023) → Lightning (ByteDance, 2024) は同じ「SDXL を 4 step 化する」目標を別チームが独立に解いた成果物。Lightning の方が品質で評価されている

00 の「同じ SD1.5 + 別画風 FT」と 01 の「同じ SDXL アーキ + LoRA 蒸留」を見比べると、**diffusion model の派生**には大きく 2 系統 (style FT / efficiency distillation) があり、**どちらも LoRA 配布が現代の標準**になっていることが分かります。

### 🧹 メモリ整理（§9 の前に Base / Lightning を解放）

ここまでの base SDXL (`pipe`) と Lightning (`pipe_alt`) は §9 以降では使わない。GPU から解放してメモリを空ける。
（解放しないとモデルが積み上がり、Animagine + LoRA を載せる §10 で GPU メモリが溢れる。実測: 無解放だと L4 24GB でも §10 で OOM）


In [ ]:
# === メモリ整理: §9 の前に Base / Lightning を完全解放 ===
# §2 の base pipe と §8 の Lightning は §9 以降では使わない。GPU を空けて §10 の OOM を防ぐ。
# base pipe の部品(unet/vae/text_encoder…)を握るのは §5/§6 の便宜変数と §4 ループの obj。
# それらと本体を手放せば解放される（§4 の components 一覧はキー名表示にしてあるので、
# 巨大 dict は IPython にキャッシュされず、ここで完全に解放できる）。
import gc
_g = globals()
for _n in ("pipe", "pipe_alt", "unet_alt",
           "unet", "vae", "text_encoder", "text_encoder_2",
           "scheduler", "tokenizer", "tokenizer_2",
           "obj", "name", "cls"):
    _g.pop(_n, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"freed base/lightning -> GPU allocated {torch.cuda.memory_allocated()/1e9:.1f} GB")
else:
    print("freed base/lightning")

<a id="animagine"></a>

---
## 9. anime full FT model: Animagine XL 3.1

ここまでは SDXL Base + (Lightning LoRA / 共有 component) という構成でした。§9 では完全に違うアプローチ — **anime 専用に学習された別モデル**を読み込んで試します。

### Animagine XL 3.1 とは

- HF Hub レポジトリ: [`cagliostrolab/animagine-xl-3.1`](https://huggingface.co/cagliostrolab/animagine-xl-3.1) (Cagliostro Research Lab、2024 年公開)
- **SDXL Base 系列の anime 特化モデル**。model card の「Fine-tuned from」は **Animagine XL 3.0**、model tree は **SDXL Base 1.0 → Animagine XL 2.0 → 3.0 → 3.1** という **3 段階の継承的 FT 系譜** (SDXL Base から直接 1 段で作ったのではない)
- 月 ~167k download (HF Hub で最 popular な SDXL anime model の 1 つ)
- 位置: SD1.5 における **Anything-v4 と同じく「anime SDXL のデファクト」**
- アーキテクチャは SDXL Base と完全に同じなので `from_pretrained()` 一発で読める

### コミュニティ事情 — civitai と anime SDXL エコシステム

SD / SDXL の anime エコシステムは、Hugging Face Hub だけでなく **civitai (civitai.com)** が最大の集積地です。個人 hobbyist 〜 小規模ラボが画風 / キャラクター / コンセプトの LoRA や FT モデルを **数千〜数万単位**で公開しており、その文化が SDXL でも継承されています。

主要 anime full FT モデル (2026 年現在、いずれも community 由来):

- **Animagine XL 3.1** (cagliostrolab) ← 本ノートで使用、一般的 anime デファクト
- **Pony Diffusion V6 XL** (PurpleSmartAI) — Pony 系派生、アニメ + キャラクター強い (NSFW 文化のため lecture 配布は非推奨)
- **Illustrious XL** (OnomaAI) — anime base model 自体、後発派生の親モデル
- **NoobAI XL** — Illustrious 派生、低 step + 高品質、2025 年に急成長

各モデルは「**base SDXL から派生したが、もはや別モデル**」という位置で、追加 LoRA も「Animagine 用」「Pony 用」「Illustrious 用」と互換性が分かれているのが SDXL anime 文化の特徴です。SD1.5 時代の Anything-v4 が 1 個でデファクトを取っていたのと比べると、SDXL では **複数の派生 base が並立**しています。

### このノートで試すこと

§3 と同じ seed + 同じ「魔法使い」テーマで Animagine XL に生成させて、**画風がどう変わるか**を見ます。ただし anime model は prompt convention (Danbooru タグ形式) が違うので、prompt は anime 仕様に書き換えます。

注意: **追加 download ~14 GB**。SDXL Base と同水準で、初回は時間がかかります。

In [ ]:
ANIME_MODEL_ID = "cagliostrolab/animagine-xl-3.1"

print(f"loading anime full model: {ANIME_MODEL_ID}")
print("(初回 download は ~14 GB、回線次第で十数分〜)")
t_load_anime = time.perf_counter()

# §3 と同じ from_pretrained API — model_id を差し替えるだけ (アーキは SDXL Base と同じ)
pipe_anime = StableDiffusionXLPipeline.from_pretrained(
    ANIME_MODEL_ID,
    torch_dtype=dtype,
    use_safetensors=True,
)
pipe_anime = pipe_anime.to(device)

# Animagine 推奨 scheduler: EulerAncestralDiscreteScheduler (Euler a)
# (Base SDXL の既定 EulerDiscreteScheduler とは別物、anime 領域で好まれる)
from diffusers import EulerAncestralDiscreteScheduler
pipe_anime.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe_anime.scheduler.config)

if device == "mps":
    pipe_anime.enable_attention_slicing()

load_elapsed_anime = time.perf_counter() - t_load_anime
print(f"load done in {load_elapsed_anime:.2f} s (初回は ~14 GB download 時間を含む)")
print("pipeline       :", type(pipe_anime).__name__)
print("scheduler      :", type(pipe_anime.scheduler).__name__, "(Animagine 推奨 Euler a)")

In [ ]:
# Animagine 推奨: Danbooru タグ形式 prompt + quality tag prefix + 推奨 negative
# §3 の自然言語「魔法使い」prompt を Danbooru タグ列に翻訳
anime_prompt = (
    "masterpiece, best quality, very aesthetic, absurdres, "
    "1girl, young witch, long flowing hair, holding magic staff, "
    "glowing staff, enchanted forest, bioluminescent mushrooms, "
    "fantasy art, detailed"
)
anime_negative = (
    "nsfw, lowres, (bad), text, error, fewer, extra, missing, "
    "worst quality, jpeg artifacts, low quality, watermark, unfinished, "
    "displeasing, oldest, early, chromatic aberration, signature, "
    "extra digits, artistic error, username, scan, [abstract]"
)
# Animagine 推奨パラメータ (model card より)
anime_steps = 28
anime_guidance = 6.0

# §3 と同じ seed を再設定して、Base ↔ Animagine の比較が seed 揃いになるように
generator.manual_seed(seed)

print(f"生成中 with {ANIME_MODEL_ID}")
print(f"  steps={anime_steps}, guidance={anime_guidance}, size=1024x1024, seed={seed}")
print()
t = time.perf_counter()
result = pipe_anime(
    prompt=anime_prompt,
    negative_prompt=anime_negative,
    width=1024,
    height=1024,
    num_inference_steps=anime_steps,
    guidance_scale=anime_guidance,
    generator=generator,
)
elapsed_anime = time.perf_counter() - t
print(f"done in {elapsed_anime:.2f} s")

image_anime = result.images[0]  # pyright: ignore[reportAttributeAccessIssue]
out_path = NB_OUT_DIR / "sec9_animagine.png"
image_anime.save(out_path)
print(f"saved: {out_path}")
image_anime

**観察ポイント**

- §3 (SDXL Base、自然言語 prompt) と並べて見ると、**画風が完全に anime / イラスト系に変わっている**。同じ「魔法使い」題材なのに別世界
- prompt が Danbooru タグ形式に変わっている: 名詞句をカンマで並べる、quality tag (`masterpiece, best quality, ...`) を prefix にする、推奨 negative prompt を使う ─── これが anime model の作法
- これが「**SDXL の anime 化は full FT model で行う**」の実演。Base + LoRA で部分的に anime 寄せるよりも、anime-specific FT model を使う方が出力品質が桁違いに高い
- **生成時間 (M4 Max 実測)**: §3 Base 30 steps = 50.87 s に対し、§9 Animagine 28 steps = **66.64 s**。**steps が少ない (28 vs 30) のに ~31% 遅い** — アーキは同じでもいくつかの要因で重くなる可能性:
  - prompt 長さ: Animagine の Danbooru タグ列 + 推奨 negative prompt が §3 自然言語より長く、cross-attention 計算が増える
  - scheduler 差: Animagine 推奨の EulerAncestral は EulerDiscrete よりわずかに重い可能性 (ノイズ加算 step がある)
  - MPS の shader compile キャッシュや memory layout などの環境要因も寄与している可能性

- 「Anything-v4 みたいなものは SDXL でいうと?」の答えは **Animagine XL 3.1 (or Pony / Illustrious / NoobAI 等)** = full FT model 配布、というのが SDXL 時代の構図
- なお §8 で見た **Lightning LoRA は Animagine XL にも乗せられる** (UNet 互換) → 「Anime 画風 + 4-step 高速生成」というのが civitai で popular な実用パターン。本ノートでは試さないが、§10 で別の LoRA stacking を見る

<a id="loras"></a>

---
## 10. Animagine XL に style LoRA を stack する — 3 種類で画風を比較

§9 で Animagine XL を見ました。実は **anime 系 LoRA の多くは Animagine XL を base に使うこと前提で学習されている**ので、Animagine + LoRA の **stacking** が civitai で最も popular な実用パターンです。

このセクションでは [Linaqruf](https://huggingface.co/Linaqruf) が公開している **3 種類の style LoRA を順に Animagine 3.1 に当てて画風を比較**します:

### 00. HF Hub レポジトリ [`Linaqruf/style-enhancer-xl-lora`](https://huggingface.co/Linaqruf/style-enhancer-xl-lora) (詳細説明はここ、以下 01/02 は概略)

- 著者 **Linaqruf** は anime LoRA を多数公開している個人 hobbyist (Animagine XL 2.0 の作者でもある)
- model card には「**Animagine XL 2.0 を base に使うこと前提**」と明記
- 月 ~2944 download (anime LoRA としては popular)
- **ファイルサイズ ~47 MB** (Lightning LoRA 376 MB の **~1/8**、community style LoRA は数十 MB と非常に軽量なケースも多い)
- 推奨 `lora_scale=0.6`、本ノートでも同値を使う
- 効果: anime 画質向上 + SD1.5 風 'old-school' 再現 (model card より)

### 01. HF Hub レポジトリ [`Linaqruf/anime-detailer-xl-lora`](https://huggingface.co/Linaqruf/anime-detailer-xl-lora)

- **concept slider 型**で他とは仕様が違う: `lora_scale` を **-2 〜 2** で扱い、`+2` で detail 最大、`-2` で detail 最小 (model card 推奨 example は `2`)
- Linaqruf 系で最 popular (3.1k dl)

### 02. HF Hub レポジトリ [`Linaqruf/anime-nouveau-xl-lora`](https://huggingface.co/Linaqruf/anime-nouveau-xl-lora)

- 装飾的・古典 anime 美学 (modern + classic blend、model card より)
- 推奨 `lora_scale=0.6`

### 重要な注意 (全 3 LoRA 共通): **Animagine 3.1 + 2.0 向け LoRA** という非推奨組み合わせを試している

3 LoRA とも model card は「**Animagine XL 2.0 を base に使うこと前提**」。本ノートで使う base は §9 で読み込んだ **Animagine XL 3.1** なので、「学習時の base (2.0) と推論時の base (3.1) が異なる」状態:

- 同じ **Animagine 系列**ではあるので破綻はしない (互換性が大きく崩れない近縁系統)
- しかし model card example の品質と完全には一致しない (推奨されている厳密な組み合わせではない、**試験的適用**)
- 実測では 3 LoRA とも 3.1 でちゃんと動作することを確認済 (画風差が出る)

ここでは「最新の Animagine 3.1 と組み合わせて使ってみる」という実用シナリオを見せます。最高品質を求める場合は LoRA を Animagine XL 2.0 にロードするのが公式推奨。

### LoRA stacking とは

`pipe.load_lora_weights()` は何度でも呼べて、複数 LoRA を組み合わせて使えます。civitai 実用文化は:

- **base anime model** (Animagine / Pony / Illustrious) を選ぶ
- その上に **style LoRA** (画風)、**character LoRA** (特定キャラ)、**concept LoRA** (構図・モチーフ) などを 1〜3 個 stack
- 各 LoRA の strength を `pipe.set_adapters([name1, name2], adapter_weights=[1.0, 0.5])` で個別調整

本ノートでは **3 LoRA を順に 1 つずつ単独 stack** (複数を同時 stack するのは API としては可能だがスコープ外)。

### 実装方針

ループ内で **LoRA ごとに pipe を組み立て直す**:

1. 各 LoRA で `pipe_anime` の text encoder × 2 / VAE / tokenizer × 2 を **リファレンス共有** (メモリ追加なし)
2. **UNet だけ `copy.deepcopy()`** して別インスタンスにする (LoRA fuse の副作用が `pipe_anime` に波及しない、3 LoRA 間でも干渉しない)
3. scheduler は新規 (Animagine と同じ Euler a)
4. `load_lora_weights(...)` + `fuse_lora(lora_scale=...)` で LoRA 適用 → 生成 → 画像保存 → memory 解放

これで 3 LoRA の出力を独立に並べて見られます。

In [ ]:
# 3 LoRA を順に試すための設定リスト。
# 全 LoRA とも model card 上は Animagine XL 2.0 ベース (Linaqruf = Animagine 2.0 の作者) で、
# 本ノートでは Animagine 3.1 に試験的適用する (推奨外組み合わせ、§10 intro 参照)。
# scale / filename / 各 LoRA の特徴は事前に WebFetch で各 model card を確認済。

LORA_CFG = [
    # 00: 既定。anime 画質向上 + SD1.5 風 'old-school' 再現 (model card 引用)
    {
        "label": "00_style-enhancer",
        "repo": "Linaqruf/style-enhancer-xl-lora",
        "ckpt": "style-enhancer-xl.safetensors",
        "scale": 0.6,  # model card 推奨値
    },
    # 01: concept slider 型 LoRA。scale range -2..2 で detail 量を制御
    {
        "label": "01_anime-detailer",
        "repo": "Linaqruf/anime-detailer-xl-lora",
        "ckpt": "anime-detailer-xl.safetensors",
        "scale": 2.0,  # slider range -2..2 のうち最大 detail (model card 推奨 example の値)
    },
    # 02: Anime Nouveau 風 (装飾・古典 anime 美学、model card より)
    {
        "label": "02_anime-nouveau",
        "repo": "Linaqruf/anime-nouveau-xl-lora",
        "ckpt": "anime-nouveau-xl.safetensors",
        "scale": 0.6,  # model card 推奨値
    },
    # 03: pastel-anime — pastel 美学 anime (model card より)
    #     注: model card は base を "Animagine XL" とだけ書き、version 明示なし。
    #         scale も明示なし → 他の Linaqruf LoRA と同じ 0.6 を仮置き
    # {
    #     "label": "03_pastel-anime",
    #     "repo": "Linaqruf/pastel-anime-xl-lora",
    #     "ckpt": "pastel-anime-xl.safetensors",
    #     "scale": 0.6,  # model card 未記載、他 LoRA からの推測
    # },
    # 04: sketch-style — monochrome / greyscale / sketch / traditional media (model card より)
    #     注: trigger keywords を prompt に含めるのが推奨 (上記 4 種)
    # {
    #     "label": "04_sketch-style",
    #     "repo": "Linaqruf/sketch-style-xl-lora",
    #     "ckpt": "sketch-style-xl.safetensors",
    #     "scale": 0.6,  # model card 推奨値
    # },
]

print(f"LoRA loop: {len(LORA_CFG)} entries")
for cfg in LORA_CFG:
    print(f"  {cfg['label']:24s} -> {cfg['repo']} (scale={cfg['scale']})")

In [ ]:
import copy
import gc
from IPython.display import display

# 各 LoRA を順に試す: pipe を組み立てて fuse → 生成 → 保存 → 表示 → 解放
print(f"3 LoRA loop on {ANIME_MODEL_ID} (steps={anime_steps}, guidance={anime_guidance}, seed={seed})")
print()

lora_results = []
for cfg in LORA_CFG:
    print(f"=== {cfg['label']} ===")
    t_total = time.perf_counter()

    # pipe を毎ループ deepcopy で構築 (Animagine の text_encoder/VAE/tokenizer は共有、UNet のみ専有)
    unet_demo = copy.deepcopy(pipe_anime.unet)
    scheduler_demo = type(pipe_anime.scheduler).from_config(pipe_anime.scheduler.config)
    pipe_demo = StableDiffusionXLPipeline(
        vae=pipe_anime.vae,
        text_encoder=pipe_anime.text_encoder,
        text_encoder_2=pipe_anime.text_encoder_2,
        tokenizer=pipe_anime.tokenizer,
        tokenizer_2=pipe_anime.tokenizer_2,
        unet=unet_demo,
        scheduler=scheduler_demo,
    )
    pipe_demo = pipe_demo.to(device)
    if device == "mps":
        pipe_demo.enable_attention_slicing()

    # LoRA load + fuse
    pipe_demo.load_lora_weights(cfg["repo"], weight_name=cfg["ckpt"])
    pipe_demo.fuse_lora(lora_scale=cfg["scale"])

    # 生成 (§9 と同じ Danbooru prompt / steps / guidance / seed)
    generator.manual_seed(seed)
    t_gen = time.perf_counter()
    out = pipe_demo(
        prompt=anime_prompt,
        negative_prompt=anime_negative,
        width=1024,
        height=1024,
        num_inference_steps=anime_steps,
        guidance_scale=anime_guidance,
        generator=generator,
    )
    elapsed_gen = time.perf_counter() - t_gen
    img = out.images[0]  # pyright: ignore[reportAttributeAccessIssue]

    # 外部 PNG 保存
    out_path = NB_OUT_DIR / f"sec10_{cfg['label']}.png"
    img.save(out_path)

    elapsed_total = time.perf_counter() - t_total
    print(f"  gen   : {elapsed_gen:.2f} s")
    print(f"  total : {elapsed_total:.2f} s (pipe 構築 + LoRA load + 生成)")
    print(f"  saved : {out_path}")
    display(img)

    lora_results.append({"label": cfg["label"], "image": img, "elapsed_gen": elapsed_gen, "elapsed_total": elapsed_total, "path": str(out_path)})

    # memory 解放
    del pipe_demo, unet_demo, scheduler_demo
    if device == "mps":
        torch.mps.empty_cache()
    gc.collect()
    print()

print("=== summary ===")
for r in lora_results:
    print(f"  {r['label']:24s} gen={r['elapsed_gen']:.2f}s saved={r['path']}")

**観察ポイント**

3 LoRA を Animagine XL 3.1 に当てた結果:

- **00 style-enhancer**: anime 画質強化、輪郭・色味がやや締まる方向
- **01 anime-detailer**: concept slider の detail+2、細部の密度が増す方向 (情報量が多めの画になる)
- **02 anime-nouveau**: 装飾的・古典 anime 美学が混ざる、modern + classic blend

3 とも **Animagine 3.1 で問題なく動作** (学習時 base = 2.0 と推論時 base = 3.1 は厳密には一致しないが、近縁系統なので破綻なし)。**画風差は明確に見える**が、最高品質を求めるなら LoRA を Animagine XL 2.0 にロードするのが公式推奨。

### LoRA stacking の一般メモ

- `lora_scale=0.6` は style/concept LoRA でよく見る推奨。slider 型 LoRA (anime-detailer のような concept slider) は別仕様 (-2..2 range)
- `fuse_lora(lora_scale=X)` で焼き込み時に重みを scale できる
- 1.0 にすると effect 強すぎ、0.0 だと LoRA 効かないのと同じ
- **生成時間 (M4 Max 実測)**: §9 Animagine 単体 66.64 s に対し、§10 (Animagine + LoRA) は **3 LoRA とも ~70 s 前後** (gen 69.46 / 70.82 / 71.26 s)。LoRA fuse 済なので推論時オーバーヘッドはほぼゼロ、+3-4 s 差は MPS の noise の範囲。「LoRA を stack しても生成は重くならない」 = `fuse_lora` の利点
- 「**Animagine + LoRA stacking**」が civitai 文化の本丸。個人作家が「自分好みの画風」を LoRA で配布し、ユーザは Animagine + 好きな LoRA 1-3 個 stack して使う、というのが日常的な使い方

### `load_lora_weights()` warning の意味

3 LoRA とも cell 実行時に下記 warning を出す:
```
No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'.
No LoRA keys associated to CLIPTextModelWithProjection found with the prefix='text_encoder_2'.
```
これは **この LoRA file には text_encoder 用の LoRA key が含まれていない** という事実を示しており、今回 LoRA が変更しているのは UNet 側だけ。Lightning LoRA (§8) と Linaqruf 系 style/concept LoRA (§10) の共通パターン。ただし **Diffusers の `load_lora_weights()` API 自体は UNet + text encoder 両方の LoRA を扱える**ため、「全 SDXL LoRA は UNet だけ」と一般化はできない (LoRA file に text encoder 用 key が含まれていれば API が両方を読み込んで適用する)。

### 重要 caveat: LoRA は learning time と inference time の base が一致しないと本来の効果は出ない

3 LoRA はすべて **Animagine XL 2.0 用に学習**された LoRA を **Animagine XL 3.1 に適用** している。**完全な品質は出ない**前提の試験的組み合わせ。3.1 ベースでちゃんと動作する事実 + Linaqruf 系列で同じ作者が 3 種類画風を提供している事実、を見せるのが本セクションの目的。

### まとめると 01 で見た 4 種類の派生

| 派生方式 | セクション | 例 (本ノート) | サイズ | 生成時間 (M4 Max 実測 ref) | 目的 |
|---|---|---|---:|---:|---|
| **efficiency LoRA on Base** | §8 | [`ByteDance/SDXL-Lightning`](https://huggingface.co/ByteDance/SDXL-Lightning) | **376 MB** | **4.88 s** (4 steps) | step 数を減らす (蒸留) |
| **anime full FT model** | §9 | [`cagliostrolab/animagine-xl-3.1`](https://huggingface.co/cagliostrolab/animagine-xl-3.1) | **~14 GB** | **66.64 s** (28 steps) | 画風を anime に完全変更 |
| **style LoRA on anime base** | §10 | [`Linaqruf`](https://huggingface.co/Linaqruf) 製 (style/concept) × 3 | **~47 MB / 個** | **~70 s / 個** (28 steps) | anime 画風を細部で強化 |
| (将来) Lightning + Animagine 組合せ | — | (本ノートでは試さない) | combo | ~10 s (見積) | anime 画風 + 4-step 高速生成 |

この 4 パターンが SDXL エコシステムでの主要な派生方式で、civitai / HF Hub に出回っているモデル / LoRA はだいたいこのどれかに当てはまります。

サイズの幅に注目: **47 MB (Linaqruf style/concept) → 376 MB (Lightning) → 14 GB (Animagine full)** と、**300 倍のレンジ**で「同じ目的 (画風変更 / 速度改善) を達成する手段が複数ある」のが SDXL エコシステムの面白いところ。

### 生成画像の保存先

このセクションで生成した 3 LoRA の画像は以下にも保存されます (notebook 外で diff / 拡大確認に便利):

- `outputs/01_sdxl_base_intro/sec10_00_style-enhancer.png`
- `outputs/01_sdxl_base_intro/sec10_01_anime-detailer.png`
- `outputs/01_sdxl_base_intro/sec10_02_anime-nouveau.png`

<a id="summary"></a>

---
## まとめ

### SD1.5 (00) と SDXL Base (01) の対比

| ステップ | 処理 | SD1.5 (00) | SDXL Base (01) |
|---|---|---|---|
| tokenize | prompt → token id 列 | tokenizer × 1 (CLIP-L 用 BPE) | **tokenizer × 2** (CLIP-L 用 / bigG 用の 2 系統。ただし SDXL Base 1.0 公式 repo では両者とも同じ CLIPTokenizer = 同一 vocab/BPE) |
| text encode | token 列 → embedding | text_encoder × 1 → (1, 77, **768**) | **text_encoder × 2** → concat (1, 77, **2048**) + pooled (1, 1280) |
| denoise | T 回 noise を引く | UNet 860 M (cross-attn 768) | **UNet 2.6 B (cross-attn 2048)** + text-time 追加条件 |
| decode | latent → 画像 | VAE: (1, 4, **64, 64**) → (3, **512, 512**) | VAE: (1, 4, **128, 128**) → (3, **1024, 1024**) |

**SDXL Base の特徴を一行で**: SD1.5 と同じ骨格 (CLIP text + UNet + VAE) のまま、**text encoder を 2 系統に増やし、UNet を 3 倍重くし、解像度を 4 倍** にした拡張版。

### 次に向けて

SD1.5 (00) と SDXL Base (01) を並べると、「latent diffusion model の基本骨格は同じで、各部品を太らせていく」という設計の流れが見えます。今後のノートブックでは:

- **FLUX.1-schnell** (script 05): text encoder が T5 + CLIP に変わる、UNet ではなく DiT (Diffusion Transformer) を使う、より新しい構造
- **Stable Diffusion 3.5 Medium** (script 06): SDXL の延長線上で MMDiT という新しい backbone
- **Qwen-Image** (script 07): Alibaba 系、別系統の DiT

といった、より新しいモデル系統での「内部の見方」を扱っていく予定です。